## Example 1: Basic Compression Codecs

In [ ]:
import numcodecs
import numpy as np

print("Example 1: Basic Compression Codecs")

# Create sample data
data = np.random.rand(1000).astype('float32')
original_bytes = data.tobytes()
print(f"Original size: {len(original_bytes)} bytes")

# Test different codecs
codecs = [
    ('Zlib', numcodecs.Zlib()),
    ('Blosc', numcodecs.Blosc()),
    ('LZ4', numcodecs.LZ4())
]

for name, codec in codecs:
    compressed = codec.encode(original_bytes)
    ratio = len(original_bytes) / len(compressed)
    print(f"{name}: {len(compressed)} bytes, {ratio:.2f}x")

print("\nExample 1 completed successfully")

## Example 2: Delta Encoding and Filters

In [ ]:
import numcodecs
import numpy as np

print("Example 2: Delta Encoding and Filters")

# Create time series data
data = np.cumsum(np.random.randn(500) * 0.1) + 100
bytes_data = data.astype('float64').tobytes()

# Apply Delta filter
delta = numcodecs.Delta(dtype='float64')
encoded = delta.encode(bytes_data)
decoded = delta.decode(encoded)

print(f"Original: {len(bytes_data)} bytes")
print(f"Delta encoded: {len(encoded)} bytes")

# Verify
recovered = np.frombuffer(decoded, dtype='float64')
print(f"Max error: {np.max(np.abs(recovered - data))}")

print("\nExample 2 completed successfully")

## Example 3: Blosc Configuration

In [ ]:
import numcodecs
import numpy as np

print("Example 3: Blosc Configuration")

# Create test data
data = np.random.rand(5000).astype('float32').tobytes()

# Different Blosc configurations
configs = [
    ('Default', numcodecs.Blosc()),
    ('LZ4HC', numcodecs.Blosc(cname='lz4hc', clevel=9)),
    ('ZSTD', numcodecs.Blosc(cname='zstd', clevel=3))
]

for name, codec in configs:
    compressed = codec.encode(data)
    ratio = len(data) / len(compressed)
    print(f"{name}: {ratio:.2f}x compression")

print("\nExample 3 completed successfully")

## Example 4: Codec Registry and Configuration

In [ ]:
import numcodecs
import json

print("Example 4: Codec Registry and Configuration")

# Show registry
registry = list(numcodecs.registry.codec_registry.items())
print(f"Total codecs: {len(registry)}")
print(f"Sample: {[name for name, _ in registry[:5]]}")

# Create from config
config = {"id": "blosc", "cname": "lz4", "clevel": 5}
codec = numcodecs.get_codec(config)
print(f"\nCreated: {type(codec).__name__}")
print(f"Config: {json.dumps(codec.get_config())}")

print("\nExample 4 completed successfully")

## Example 5: Codec Pipeline

In [ ]:
import numcodecs
import numpy as np

print("Example 5: Codec Pipeline")

data = np.arange(1000, dtype='uint16')
original = data.tobytes()

# Pipeline: Delta -> Shuffle -> Compress
pipeline = [
    numcodecs.Delta(dtype='uint16'),
    numcodecs.Shuffle(elementsize=2),
    numcodecs.LZ4()
]

# Encode
current = original
for codec in pipeline:
    current = codec.encode(current)
print(f"Compressed: {len(original)} -> {len(current)} bytes")

# Decode
for codec in reversed(pipeline):
    current = codec.decode(current)
matched = np.array_equal(np.frombuffer(current, dtype='uint16'), data)
assert matched, "the Delta -> Shuffle -> LZ4 pipeline did not reproduce its input"
print(f"Match: {matched}")

## Example 6: Lossless Round-Trips (asserted)

In [ ]:
import itertools
from multiprocessing.pool import ThreadPool

import numpy as np
from numcodecs import LZ4, Blosc, Zstd, blosc

rng = np.random.default_rng(20260923)
arrays = (rng.integers(-1000, 1000, 100_000, dtype="i4"), rng.normal(size=50_000))
cnames = ("lz4", "lz4hc", "zstd", "zlib", "blosclz")
missing = set(cnames) - set(blosc.list_compressors())
assert not missing, f"Blosc was built without compressor(s) {sorted(missing)}"
shuffles = (Blosc.NOSHUFFLE, Blosc.SHUFFLE, Blosc.BITSHUFFLE)
codecs = [Blosc(cname=c, clevel=5, shuffle=s) for c, s in itertools.product(cnames, shuffles)]
cases = list(itertools.product(codecs + [LZ4(), Zstd(level=3)], arrays))

def mismatch(case):
    codec, array = case
    decoded = np.frombuffer(codec.decode(codec.encode(array)), dtype=array.dtype)
    return None if np.array_equal(decoded, array) else f"{codec!r} on {array.dtype}"

bad = [m for m in map(mismatch, cases) if m]
with ThreadPool(8) as pool:
    bad += [f"[worker thread] {m}" for m in pool.map(mismatch, cases) if m]
assert not bad, f"{len(bad)} of {2 * len(cases)} round-trips did not reproduce their input: {bad}"
print(f"Example 6: {len(cases)} codec x array round-trips reproduced, main thread and 8 worker threads")